# Assignment

## Brief

Write the Python codes for the following questions.

## Instructions

Paste the answer as Python in the answer code section below each question.

### Question 1

Question: Implement a simple Thrift server and client that defines a `Student` struct with fields `name` (string), `age` (integer), and `courses` (list of strings). Include a service `School` with a method `enrollCourse` that takes a `Student` record and a course name, adds the course to the student's course list, and returns the updated `Student` record.

Answer:

**Thrift schema (student.thrift)**

In [15]:
%%writefile student.thrift
struct Student {
    1: required string name,
    2: required i32 age,
    3: required list<string> courses
}

service School {
    Student enrollCourse(1: required Student student, 2: required string course)
}


Overwriting student.thrift


**Thrift server (student_server.py)**

In [16]:
%%writefile student_thrift_server.py
# Import thriftpy2, the Python library that can read Thrift schema files
import thriftpy2

# Load the student.thrift schema/rulebook
# This creates Python objects from the Thrift definitions
student_thrift = thriftpy2.load(
    "student.thrift",
    module_name="student_thrift"
)

# Import the helper that creates an RPC server
from thriftpy2.rpc import make_server


# Create the actual Python implementation of the School service
class School(object):

    # This method must match the method name in student.thrift: enrollCourse
    def enrollCourse(self, student, course):

        # Add the new course into the student's courses list
        student.courses.append(course)

        # Return the updated student back to the client
        return student


# Create the Thrift server
# student_thrift.School = the service definition from the schema
# School() = our actual Python implementation above
server = make_server(
    student_thrift.School,
    School(),
    client_timeout=None
)

print("Starting and running the Thrift server...")
print("Press Ctrl+C to stop the server.")

# Start the server and keep it waiting for client requests
server.serve()

Overwriting student_thrift_server.py


Run the thrift server

**Thrift client (student_client.py)**

In [18]:
%%writefile student_client.py
# Import thriftpy2 so the client can read the same Thrift schema as the server
import thriftpy2

# Load the student.thrift schema/rulebook
# IMPORTANT: thriftpy2 module_name must end with "_thrift"
student_thrift = thriftpy2.load(
    "student.thrift",
    module_name="student_client_thrift"
)

# Import the helper that creates a Thrift RPC client
from thriftpy2.rpc import make_client


# Create a client object for the School service
# This lets the client call the remote enrollCourse method on the server
school = make_client(
    student_thrift.School,
    timeout=None
)


# Create a Student object based on the Student struct in student.thrift
student = student_thrift.Student(
    name="Kenny",
    age=35,
    courses=["Python", "SQL"]
)

# Show the student's courses before calling the server
print("Before enrolling:")
print(student.courses)


# Call the remote enrollCourse method on the server
# This sends the student and the new course to the server
# The server adds the course and returns the updated student
student = school.enrollCourse(
    student,
    "Data Engineering"
)


# Show the student's courses after the server updates the record
print("After enrolling:")
print(student.courses)

Overwriting student_client.py


### Question 2

Question: Implement a simple Protocol Buffers server and client that defines a `Book` message with fields `title` (string), `author` (string), and `page_count` (integer). Include a service `Library` with a method `checkoutBook` that takes a `Book` message and returns the same `Book` message.

Answer:

**Protobuf schema (book.proto)**

In [19]:
%%writefile book.proto
syntax = "proto3";

// Define a Book message/data object
// This is like saying: a Book has a title, author, and page_count
message Book {
    string title = 1;
    string author = 2;
    int32 page_count = 3;
}

// Define a Library service
// This service has one remote method called checkoutBook
service Library {
    rpc checkoutBook(Book) returns (Book) {}
}

Writing book.proto


Run

**Protobuf server (book_server.py)**

In [20]:
%%writefile book_server.py
# Import tools for running the server with worker threads
from concurrent import futures

# Import the gRPC library
import grpc

# Import the generated gRPC helper code from book.proto
# This file was created by the protoc compiler
import book_pb2_grpc


# Create the actual Python implementation of the Library service
# LibraryServicer comes from the generated book_pb2_grpc.py file
class Library(book_pb2_grpc.LibraryServicer):

    # This method must match the rpc method name in book.proto: checkoutBook
    # self = the current Library server object
    # request = the Book message sent by the client
    # context = gRPC request context/details
    def checkoutBook(self, request, context):

        # For this assignment, simply return the same Book message
        # No modification needed
        return request


# Create a gRPC server
# max_workers=2 means the server can handle requests using up to 2 worker threads
server = grpc.server(
    futures.ThreadPoolExecutor(max_workers=2)
)

# Attach our Library implementation to the gRPC server
book_pb2_grpc.add_LibraryServicer_to_server(
    Library(),
    server
)

# Start the server on localhost port 50052
# "insecure" means no SSL/encryption, okay for local lesson demo
server.add_insecure_port("[::]:50052")

print("Starting and running the Book gRPC server...")
print("Press Ctrl+C to stop the server.")

# Start the server
server.start()

# Keep the server alive and waiting for client requests
server.wait_for_termination()

Writing book_server.py


Run

**Protobuf client (book_client.py)**

In [21]:
%%writefile book_client.py
# Import the gRPC library
import grpc

# Import the generated Book message code from book.proto
# This gives us book_pb2.Book
import book_pb2

# Import the generated gRPC client helper code
# This gives us book_pb2_grpc.LibraryStub
import book_pb2_grpc


# Open a client connection to the server
# localhost = this machine
# 50052 = the same port used by book_server.py
with grpc.insecure_channel("localhost:50052") as channel:

    # Create a client-side remote control for the Library service
    stub = book_pb2_grpc.LibraryStub(channel)

    # Create a Book message using the generated Book class
    book = book_pb2.Book(
        title="The Fellowship of the Data Goblin",
        author="J. R. R. Stackoverflow",
        page_count=423
    )

    # Print the book before sending it to the server
    print("Before checkout:")
    print(book)

    # Call the remote checkoutBook method on the server
    # This sends the Book to the server and receives a Book back
    checked_out_book = stub.checkoutBook(book)

    # Print the Book returned by the server
    print("After checkout:")
    print(checked_out_book)

Writing book_client.py
